# NLP with machine learning in Python

Welcome to the second tutorial, "Natural Language Processing in Python." In this tutorial, we will see how to load the dataset in Python and do a first exploration of the available data. Even if the authors of the dataset did some work to preprocess and structure the data, we usually still need to adapt it to our needs. We will thus save it in a more convenient data structure for our use case. We will also prepare features to make the data accessible to our machine-learning algorithm. Lastly, we will train a model to predict the political ideology of the texts.


We will use a dump of debate.org for this part of the tutorial. You can find and download it [here](http://www.cs.cornell.edu/~esindurmus/ddo.html). In JupyterHub, the data is already downloaded. If you download the dataset yourself, you have at least two files: the `debate.json` file and the `users.json` file. This notebook expects the two files to be in the `path` directory, so you should change the variable if you work on your own machine.

We will also use the following packages. Please make sure to install them in your Python environment.
- [spaCy](https://spacy.io/usage)
- [scikit-learn](https://scikit-learn.org/stable/install.html)

If you are using JupyterHub provided to you by the LUIS-Cluster for the labs, the necessary packages have been preinstalled. _If your local environment differs from the JupyterHub, we cannot guarantee that your solution is valid $-$ always run your code on the Hub before submission._

### 2.1 Data loading and exploration

In [ ]:
# Since our data is saved in json format, we use the json module to load the data easily.
import json

path = "/bigwork/nhwpquec/argumentation-technology-26s/tutorial/"

with open(path+"debates.json", "r") as f:
    debates = json.load(f)

with open(path+"users.json", "r") as f:
    users = json.load(f)

The data is now stored in the two variables `debates` and `users`. Both are of the type `dict`. Let's have a quick look at the structure of the data. If we print the keys of the debates dict, we can see that the details of each debate on the portal can be accessed by its title.

In [ ]:
# Transform the dict keys to a list to display only a few debates
debate_keys = list(debates.keys())
print("\n".join(debate_keys[:10]))

Each debate object is also stored in a dictionary, which we can print to determine which properties are present. Let's do that for one debate in the data and have a closer look at the text property (each of the debates is separated into rounds, which in turn has one argument/text from each side; so we need to use the `rounds` property to inspect the text). For convenience, I save the chosen debate into a new variable.

In [ ]:
any_debate = list(debates.values())[42]
print(f"Debate properties: {any_debate.keys()}")

In [ ]:
# Print the second argument of the second debate round
print(f"Debate text: {any_debate['rounds'][1][1]['text']}")

Now that we know a thing or two about the structure of the debates, let's do the same for the user data.

In [ ]:
# Since there are so many keys, we better transform them to a list first
# to display only a fraction of them.
user_keys = list(users.keys())
print(user_keys[:10])

In [ ]:
any_user = user_keys[40]
print(f"Username: {any_user}")
print("User properties:")
users[any_user]

To make later tasks with the data a bit easier, let's collect the necessary information into a `pandas` `DataFrame`. We will attempt to find each debate text and the political ideology of its author (we assume here that the political ideology of the text is reflected in the political ideology of its author). To keep things simple, we will focus on two main ideologies: Conservative and Liberal (those are the most prominent groups in the data).

Let's iterate over all debates, all rounds in those debates and all arguments in those rounds and save the respective texts, together with the political ideology of its author.

In [ ]:
debate_data = []

for key, debate in debates.items():
    # Sometimes, the users of the debate didn't exist anymore at the time the
    # data was collected. If this is the case, simply skip this debate.
    try:
        # Get the user information from the user data we loaded previously
        user1 = users[debate["participant_1_name"]]
        user2 = users[debate["participant_2_name"]]
    except KeyError:
        continue
    
    # For each round in this debate...
    for debate_round in debate["rounds"]:
        # For each argument in this round...
        for argument in debate_round:
            # Save the text and find the political ideology of the author
            # To identify the author of the current argument, we use the "side"
            # property and compare it to the participant identifiers of the
            # debate (we don't have user names available here).
            debate_data.append({
                "debate_text": argument["text"],
                "political_ideology": user1["political_ideology"]
                                      if argument["side"] == debate["participant_1_position"]
                                      else user2["political_ideology"]
            })

Now, let's filter the debates for the political ideology of the author to only include the two mentioned groups above...

In [ ]:
political_ideologies = ["Conservative", "Liberal"]
filtered_debates = list(filter(
    lambda x: x["political_ideology"] in political_ideologies,
    debate_data))

...and save the result into a `pandas` `DataFrame`. This will make following steps a bit easier and faster, as `DataFrames` are more computationally and memory efficient than our `dict` data structure.

In [ ]:
import pandas as pd

debates_df = pd.DataFrame(
    columns=["debate_text", "political_ideology"],
    data=filtered_debates)

debates_df

We can now also easily sample the data for each political ideology and extract the same number of samples to have a more or less stratified sample. For the sake of time, we will limit the number of samples for this tutorial to the first 5000. This will make the preprocessing and feature extraction a lot faster. If you have more time (or computational power), you can try to do it with more samples. Let's double-check that by printing the number of entries in each `DataFrame`. Also, we should combine the two into a single `DataFrame`.

In [ ]:
conservative_arguments = debates_df[
    debates_df.political_ideology == "Conservative"].sample(n=2500)
liberal_arguments = debates_df[
    debates_df.political_ideology == "Liberal"].sample(n=2500)

# Combining both samples into one dataframe
debates_df = pd.concat([conservative_arguments,liberal_arguments])

# Have a look of the length of each group
print(f"Conservative arguments: {liberal_arguments.shape}")
print(f"Liberal arguments: {liberal_arguments.shape}")
print(f"Combined arguments: {debates_df.shape}")

### 2.2 Preprocessing using spaCy & Extraction of textual features

Now that we have structured that data into a more convenient format, we will take the just extracted texts and try to clean them a bit. We will use `spaCy` library to tokenize and clean the texts. Afterwards, we will try to extract meaningful features from it, which we can use to train a machine learning algorithm and predict the political ideology of a text.

Let's start by importing spaCy and loading the language model. You can use the cell below to download it to your own machine.

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

The `nlp` object now allows us to parse English text with the language model and tokenize it. Let's define a function where a given text is taken as a parameter, and the cleaned version is returned as a list of tokens.

"Cleaning" or preprocessing a text can mean different things. In our case, we want to remove punctuation, whitespace-like tokens, URLs, and stopwords. While stopwords can, in some instances, provide important contextual information, especially when working with large neural networks, they can also introduce noise. In our case, stopwords just increase the feature space too much, so we remove it.

In [ ]:
def clean_text(text: str) -> list:
    # Parse the text using the English language model
    # The returned object is an iterator over all tokens
    parsed_text = nlp(text)
    # Initialize a list which will later hold the tokens of the text
    tokenized_clean_text = []
    
    # For each token in the text...
    for token in parsed_text:
        # If the token is _not_ one of the following, append it to
        # the final list of tokens; continue otherwise
        if (not token.is_punct and  # Punctuation
                not token.is_space and  # Whitespace of any kind
                not token.like_url and # Anything that looks like an url
                not token.is_stop):  # Stopwords
            tokenized_clean_text.append(token.text.lower())
    
    # Return the list of clean tokens for this text
    return tokenized_clean_text

Now let's apply this cleaning function to all texts in the `DataFrame`. We can use the `.apply()`-function to do so and save the result in a new column. Depending on your processor and included samples, this might take some time to complete.

In [ ]:
debates_df["cleaned_text"] = debates_df["debate_text"].apply(clean_text)

We can now start with extracting features from the texts. As you may know, the Bag-of-Words method is a common approach to do that. We simply consider each unique word in our text collection as a feature dimension and the number of times that word appears in a certain text $t_j$ as the value. Thus, a Bag-of-Words feature vector for one specific text might look like the following:

$v_i = \begin{pmatrix}\# word_1\\\# word_2\\\# word_3\\\vdots\\\# word_n\end{pmatrix}$ and, more specifically $v_j = \begin{pmatrix}27\\42\\23\\\vdots\\13\end{pmatrix}$ for a text $t_j$

While not the most advanced feature representation, it is simple to understand and is often used as a simple first baseline for more complex features. So let's transform each debate into Bag-of-Words feature vectors using the `scikit-learn` library. You can find the documentation for the `CountVectorizer` [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# The CountVectorizer expects the documents as complete strings
# so we need to join our tokens back together
documents = [" ".join(document) for document in debates_df.cleaned_text.values]

# Initialize the vectorizer
vectorizer = CountVectorizer()

# Fit the vectorizer's vocabulary on the data
# and transform the input to vectors/features; save the result to `X`
x = vectorizer.fit_transform(documents)

Now that we have the feature vectors for our classification, we also need to encode our labels since they are still only present as strings (and our machine-learning model won't be able to process those). To do that, we use the `LabelBinarizer` class.

In [ ]:
from sklearn.preprocessing import LabelBinarizer

# Initialize the encoder
lb = LabelBinarizer()

# The the encoder to the dictionary and transform the labels
# into numbers
y = lb.fit_transform(debates_df.political_ideology.values)

# Reshape the data into a one-dimensional list
y = y.reshape(len(y),)

In the last preparation step, we need to split our data into training and test sets. Again, we can use the `scikit-learn` library for that.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y)

###  2.3 - Training and using a classifier

Finally, we can move on to the actual machine learning. We will use the previously prepared data, train a classifier and tune it by finding better hyperparameter values. We will also look at how to evaluate the results and how to do a cross-fold validation.

First, let's import the classifier. We will again use the `scikit-learn` library for this. We choose a simple SVM classifier for this task, as it is fast and might just work for our use case. Since the library has a lot of different classifiers, all with the same API, you might also try others on your own. You can find all classifiers in the [`scikit-learn` documentation](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html).

In [ ]:
from sklearn.svm import SVC

# Initialize the SVM
clf = SVC(gamma="scale")

# Start the training on the training data
clf.fit(x_train, y_train)

We can simply use the trained classifier to predict labels for the test dataset and save the predictions to a variable. This way, we can later evaluate the results.

In [ ]:
y_pred = clf.predict(x_test)

We will use the F1 score implementation of the `scikit-learn` library to evaluate the predictions.

In [ ]:
from sklearn.metrics import f1_score

# Evaluate the predictions and print the result
print(f1_score(y_test, y_pred))

If we want more reliable evaluation results for our chosen classifier, we can utilize the cross-validation method of the `scikit-learn` library. It will train and evaluate the model five times on different data splits.

In [ ]:
from sklearn.model_selection import cross_val_score

print(cross_val_score(clf, x, y, scoring="f1", cv=5))

Additionally, it can make sense to do a hyperparameter optimization for our classifier. Again, the `scitkit-learn` library provides some easy-to-use functions to start a grid or randomized search. Explaining these concepts is out of the scope of this tutorial, but you can find additional information [here](https://scikit-learn.org/stable/modules/grid_search.html). Also, you can try to use a more advanced method, namely, a _Bayesian optimization_, for which you can find additional information [here](http://papers.nips.cc/paper/4522-practical-bayesian-optimization-of-machine-learning-algorithms) and a Python library implementing it [here](https://github.com/fmfn/BayesianOptimization).

In the most basic sense, we provide the method with parameters (or ranges of parameters) for the classifier. The grid search below will then try all possible parameter combinations (training different models with each) and tell us which performed best.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Defining the grids that should be searched
param_grids = [
    {
        "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "gamma": ["scale"],
        "kernel": ["linear", "poly"]
    },
    {
        "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "gamma": [0.001, 0.0001],
        "kernel": ["rbf"]
    }
]

# Initializing a new classifier
grid_clf = SVC()

In [ ]:
# Initializing the grid search class
grid_search = GridSearchCV(
    grid_clf, param_grid=param_grids, cv=5, n_jobs=8, scoring="f1")

# Starting the grid search
grid_search.fit(x, y)

# Print the best parameter combination
print(grid_search.best_params_)

In [ ]:
print(
    cross_val_score(
        SVC(**grid_search.best_params_),
        x,
        y,
        scoring="f1",
        cv=5))

And that's it. Of course, we could try to improve the results further by using a different classifier like a multi-layer perceptron or more complex neural networks. Furthermore, we could choose other input features, such as Tfidf vectors or word embeddings. But the general steps described here will stay the same.

Apart from that, this is the end of this introductory tutorial. If you have any questions, feel free to ask us.

### Further references

1. https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html